In [ ]:
import os
import torchaudio
from scipy import signal
import numpy as np
import cv2
import csv
import matplotlib.pyplot as plt
import csv
import shutil
import torch

from data_converter import DataConverter

import torchvision.transforms as T

In [ ]:
#file_path = "~/AudioCounting/preprocessing/audio_strong/TEST.wav"

# file_path = "/scratch/local/hdd/hani/bbc_clocks/audio/07022026.wav"
file_path = "/scratch/local/hdd/hani/bbc_clocks/audio/07016211.wav"
# file_path = "/scratch/local/hdd/hani/bbc_clocks/audio/07016222.wav"

# file_path = "/scratch/local/hdd/hani/heartbeats/wav/e01914.wav"
# file_path = "/scratch/local/hdd/hani/heartbeats/wav/d0030.wav"
# file_path = "/scratch/local/hdd/hani/heartbeats/wav/d0015.wav"

# file_path = "/scratch/local/hdd/hani/dolphins/test_padded/PulseTrain_213.wav"
# file_path = "/scratch/local/hdd/hani/dolphins/test_padded/PulseTrain_037.wav"
# file_path = "/scratch/local/hdd/hani/dolphins/test_padded/PulseTrain_009.wav"

# file_path = "/scratch/local/ssd/hani/RS/wav/train/000010.wav"
# file_path = "/scratch/local/ssd/hani/RS/wav/train/000193.wav"
# file_path = "/scratch/local/ssd/hani/RS/wav/train/024609.wav"

# file_path = "/scratch/local/ssd/hani/RSN/wav/train/000397.wav"
# file_path = "/scratch/local/ssd/hani/RSN/wav/train/000567.wav"
# file_path = "/scratch/local/ssd/hani/RSN/wav/train/012000.wav"

# file_path = "/scratch/local/ssd/hani/RVN/wav/train/004570.wav"
# file_path = "/scratch/local/ssd/hani/RVN/wav/train/010000.wav"
# file_path = "/scratch/local/ssd/hani/RVN/wav/train/012814.wav"

# file_path = "/scratch/local/ssd/hani/RSN/wav/train/000567.wav" #for process figures

y, sr = torchaudio.load(os.path.expanduser(file_path))

if "dolphins" in file_path:
    y /= y.abs().max()

if "clocks" in file_path:
    item = file_path.split("/")[-1]
    #get start_time from csv file
    with open("/users/hani/AudioCounting/preprocessing/bbc_clocks/bbc_clocks.csv", "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["location"] == item:
                start_time = float(row["start_time"])
                break

    #remove start_time seconds from the beginning of the audio
    start_sample = int(start_time * sr)
    y = y[:, start_sample:]

if sr != 16000:
    print(f"Resampling from {sr} Hz to 16000 Hz")
    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
    y = resampler(y)
    sr = 16000

print(f"Length of audio: {y.shape[1] / sr:.2f} seconds")
y = y.numpy()


waveform = y.mean(axis=0) #* 5
_, _, spectrogram = signal.spectrogram(waveform, sr, nperseg=512, noverlap=256)
spectrogram = np.log(spectrogram + 1e-7)

converter = DataConverter()
spectrogram = converter.apply_histogram_equalisation(spectrogram, method="global")

print(spectrogram.shape)
print(spectrogram.min(), spectrogram.max())

if spectrogram.ndim == 3:
    spectrogram = spectrogram.mean(axis=0)


time_axis = np.linspace(0, len(waveform) / sr, len(waveform))

# Create subplots
fig, (ax_wave, ax_spec) = plt.subplots(2, 1, figsize=(5, 5), sharex=True, 
                                       gridspec_kw={'height_ratios': [1, 2]})

ax_wave.plot(time_axis, waveform, color='#1f77b4', linewidth=0.5)
ax_wave.set_ylabel("Amplitude")
ax_wave.set_ylim(-1, 1)
ax_wave.grid(True, alpha=0.3)

img = ax_spec.imshow(spectrogram, origin='lower', aspect='auto', cmap="Blues",
                     extent=[0, time_axis[-1], 0, sr/2000])

ax_spec.set_ylabel("Frequency (kHz)")
ax_spec.set_xlabel("Time (seconds)")

plt.subplots_adjust(hspace=0.1)
plt.tight_layout()
plt.show()

fig, (ax_wave, ax_spec) = plt.subplots(2, 1, figsize=(5, 5), sharex=True,
                                        gridspec_kw={'height_ratios': [1, 2]})
ax_wave.plot(time_axis, waveform, color='#1f77b4', linewidth=0.5)
ax_wave.set_ylim(-1, 1)
ax_wave.axis('off')
ax_spec.imshow(spectrogram, origin='lower', aspect='auto', cmap="magma",
                 extent=[0, time_axis[-1], 0, sr/2000])
ax_spec.axis('off')
plt.subplots_adjust(hspace=0)
plt.tight_layout()
plt.show()

torchaudio.save("TEST.wav", torch.from_numpy(y), sr)

In [ ]:
# batch create spectrograms

# input_dir = "/scratch/local/hdd/hani/audioset_eval/audio_mono/"
# output_dir = "/scratch/local/ssd/hani/audioset_eval/spec/test/"

input_dir = "/scratch/local/hdd/hani/bbc_clocks/audio/"
output_dir = "/scratch/local/hdd/hani/bbc_clocks/spec/test"


#csv_path = "/users/hani/AudioCounting/preprocessing/audio_strong/rep_labels.csv"
csv_path = "/users/hani/AudioCounting/preprocessing/bbc_clocks/bbc_clocks.csv"
csv_used = True
start_edit = True

files = os.listdir(input_dir)
if csv_used:
    #extract lines from csv file
    info_dict = {}
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            code = row["location"].replace(".wav", "")
            label = int(row["repetitions"])
            if start_edit:
                start_time = float(row["start_time"])
                info_dict[code] = [label, start_time]
            else:
                info_dict[code] = [label, 0.0]

    files = []
    for code in info_dict.keys():
        if os.path.exists(os.path.join(input_dir, code + ".wav")):
            files.append(code + ".wav")

converter = DataConverter()

no_of_files = len(files)
print(f"Processing {no_of_files} files.")
step_size = max(1, no_of_files // 500)
current_file = 1
for filename in files:

    y, sr = torchaudio.load(os.path.expanduser(os.path.join(input_dir, filename)))
    y = y.numpy()
    
    if csv_used:
        #get label + start time from info dict
        label, start_time = info_dict[filename.replace(".wav", "")]
        if start_time > 0:
            start_idx = int(start_time * sr)
            y = y[:, start_idx:]

    waveform = y.mean(axis=0)
    _, _, spectrogram = signal.spectrogram(waveform, sr, nperseg=512, noverlap=256)
    spectrogram = np.log(spectrogram + 1e-7)

    spectrogram = converter.apply_histogram_equalisation(spectrogram, method="global")

    mean = np.mean(spectrogram)
    std = np.std(spectrogram)
    spectrogram = np.divide(spectrogram - mean, std + 1e-9)

    if spectrogram.ndim == 3:
        spectrogram = spectrogram.mean(axis=0)

    if csv_used:
        filename = f"{filename.replace('.wav', '')}_{label}.npy"
    else:
        filename = f"{filename.replace('.wav', '')}.npy"

    output_path = os.path.join(output_dir, filename)
    np.save(output_path, spectrogram)

    if current_file % step_size == 0:
        print(f"Processed {current_file}/{no_of_files} files.")
    current_file += 1

In [ ]:
#visualise random spectrogram from directory
import os
import numpy as np
import matplotlib.pyplot as plt
spec_dir = "/scratch/local/ssd/hani/audioset_eval/spec/test/"
spec_dir = "/scratch/local/hdd/hani/bbc_clocks/spec/test/"
#spec_dir = "/scratch/local/ssd/hani/RVN/spec/test/"
spec_dir = "/scratch/local/hdd/hani/heartbeats/spec/"
spec_dir = "/scratch/local/hdd/hani/bbc_clocks/tssm/test/"
spec_dir = "/scratch/local/ssd/hani/RS/tssm/test/"
spec_files = os.listdir(spec_dir)
random_spec_file = np.random.choice(spec_files)
spec_path = os.path.join(spec_dir, random_spec_file)
spectrogram = np.load(spec_path)
# spectrogram = cv2.resize(spectrogram, (624, 257))
print(f"shape {spectrogram.shape}")
print(f"Visualising {random_spec_file}...")
plt.figure()
plt.imshow(spectrogram, origin='lower', aspect='equal', cmap="magma")
plt.axis('off')
plt.show()


In [ ]:
#test data converter methods

import random


converter = DataConverter(musan_options =["music","noise","speech"])
for _ in range(5):
    choice = random.choice(os.listdir("/scratch/local/ssd/hani/FSD50K/train/"))
    filepath = os.path.join("/scratch/local/ssd/hani/FSD50K/train/", choice)
    y, sr, num_repetitions, _ = converter.create_augmented_wav(filepath, converter.output_time, converter.max_repetitions)
    print("Repetitions:", num_repetitions)

    #create spectrogram
    waveform = y.mean(dim=0).cpu().numpy()
    _, _, spectrogram = signal.spectrogram(waveform, sr, nperseg=512, noverlap=256)
    spectrogram = np.log(spectrogram + 1e-7)

    spectrogram = converter.apply_histogram_equalisation(spectrogram, method="global")

    mean = np.mean(spectrogram)
    std = np.std(spectrogram)
    spectrogram = np.divide(spectrogram - mean, std + 1e-9)

    plt.figure()
    plt.imshow(spectrogram, origin='lower', aspect='equal', cmap="magma")
    plt.axis('off')
    plt.show()
print(spectrogram.shape)
